# 08 — silver_sellers
**Entrée :** `bronze.sellers` — **Sortie :** `silver.silver_sellers` (**3 095 lignes**)
**Nettoyage :** ville en minuscules sans espaces, État en majuscules, code postal gardé en **texte**.
**Indicateurs :** aucun (le PDF ne trouve aucune anomalie de qualité sur cette table).

In [0]:
CATALOG = "workspace"
BRONZE = f"{CATALOG}.bronze"
SILVER = f"{CATALOG}.silver"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER}")


def scalar(sql):
    return spark.sql(sql).collect()[0][0]


def check(label, obtenu, attendu):
    ok = obtenu == attendu
    print(("✅ OK     " if ok else "❌ ÉCART  ") + f"{label} : obtenu = {obtenu} | attendu = {attendu}")

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {SILVER}.silver_sellers AS
SELECT
    seller_id,
    CAST(seller_zip_code_prefix AS STRING) AS seller_zip_code_prefix,
    LOWER(TRIM(seller_city))  AS seller_city,
    UPPER(TRIM(seller_state)) AS seller_state,
    _loaded_at AS _bronze_loaded_at,
    current_timestamp() AS _silver_loaded_at
FROM {BRONZE}.sellers
""")
print("silver_sellers créée")

silver_sellers créée


In [0]:
t = f"{SILVER}.silver_sellers"
check("lignes", scalar(f"SELECT COUNT(*) FROM {t}"), 3095)
check("seller_id doublons", scalar(f"SELECT COUNT(*) - COUNT(DISTINCT seller_id) FROM {t}"), 0)
check("seller_id NULL", scalar(f"SELECT COUNT(*) FROM {t} WHERE seller_id IS NULL"), 0)
check("codes postaux de longueur != 5", scalar(f"SELECT COUNT(*) FROM {t} WHERE LENGTH(seller_zip_code_prefix) <> 5"), 0)

✅ OK     lignes : obtenu = 3095 | attendu = 3095
✅ OK     seller_id doublons : obtenu = 0 | attendu = 0
✅ OK     seller_id NULL : obtenu = 0 | attendu = 0
✅ OK     codes postaux de longueur != 5 : obtenu = 0 | attendu = 0


## R13 : effet des accents sur les villes (facultatif, ne fait jamais échouer le notebook)

In [0]:
try:
    spark.sql(f"""
        SELECT COUNT(DISTINCT seller_city) AS villes_avec_accents,
               COUNT(DISTINCT translate(seller_city,
                   'áàâãäéèêëíìîïóòôõöúùûüç', 'aaaaaeeeeiiiiooooouuuuc')) AS villes_sans_accents
        FROM {SILVER}.silver_sellers
    """).show()
except Exception as e:
    print("Cellule facultative ignorée :", str(e)[:200])

+-------------------+-------------------+
|villes_avec_accents|villes_sans_accents|
+-------------------+-------------------+
|                611|                611|
+-------------------+-------------------+



## À documenter (pas une erreur)
Les 10 premiers vendeurs réalisent environ 13 % du chiffre d'affaires. C'est une observation pour l'analyse.

## Pour ton journal (à rédiger avec tes mots)
1. Pourquoi `TRY_CAST` plutôt que `CAST` ?
2. Pourquoi on garde les lignes anormales avec un indicateur `flag_…` ?
3. Pourquoi Silver a le même nombre de lignes que Bronze ?